# Within-module interaction, one random split

A module's own knockouts cannot interact with themselves — `K_0*K_0` collapses
to `K_0` — so a within-module interaction is not estimable directly. This
notebook splits a module's knockout genes at random into two halves, makes each
half its own covariate, and fits

```
y ~ half1 + half2 + half1*half2
```

so that the two knockouts of a same-module double land in different variables
and their product is a genuine interaction.

**One module and one iteration per run.** Set `GUIDE_GROUP` and `ITERATION`
below and execute; the twenty-five files behind the published figures were made
by editing those two numbers and re-running. `05_AverageSameModuleInteractionSplits.ipynb`
averages the five splits of each module.

## Setup

In [ ]:
%load_ext rpy2.ipython

import random

import scanpy as sc
import pandas as pd
import anndata2ri
from rpy2.robjects import numpy2ri, pandas2ri

# The %%R cell below receives pandas frames; these converters are what the
# libraries.py star-import used to activate.
numpy2ri.activate()
pandas2ri.activate()
anndata2ri.activate()

DATASET    = "/home/eraslab1/Projects/E3Ligase/analysisSingle/Notebooks/CombinatorialPerturbations/dataset"
MODULE_CSV = "/home/eraslab1/Projects/E3Ligase/analysisSingle/TextFiles/ME_GuideModules_leiden_6_Modules.csv"

GUIDE_GROUP = 0      # modules 0-4; module 5 has too few same-module doubles
ITERATION   = 1      # 1-5

OUT_FILE = f"outputs/ComboEffects_doublesResample_KO_{GUIDE_GROUP}_{ITERATION}.rds"

## The random half-split

:::{warning}
The original did not seed this, so it drew a fresh half on every run and the
twenty-five published files cannot be regenerated — the saved files are the
only record of which genes went into which half. A seed is set here so that
future runs are at least reproducible, and the genes are sorted before
sampling because iteration order over a Python set is not stable between
processes. This makes the notebook deterministic going forward; it does **not**
recover the original splits.
:::

In [ ]:
guideModulesN = pd.read_csv(MODULE_CSV, index_col=0)
moduleGenes = set(guideModulesN.loc[guideModulesN.GuideGroup == GUIDE_GROUP, "GuideName"])

random.seed(GUIDE_GROUP * 100 + ITERATION)
group1 = set(random.sample(sorted(moduleGenes), round(len(moduleGenes) / 2)))
group2 = moduleGenes - group1

group1 = ["GENE_" + str(x) + "_" for x in group1]
group2 = ["GENE_" + str(x) + "_" for x in group2]

print(f"module {GUIDE_GROUP}: {len(moduleGenes)} genes -> {len(group1)} + {len(group2)}")

## Design

Single-knockout cells carrying exactly one of this module's guides — or a
control guide — are labelled by which half they fall in. Same-module doubles
carry both halves by construction.

In [ ]:
adataSingles = sc.read(f"{DATASET}/adataTrainSingles.h5ad")
adataDoubles = sc.read(f"{DATASET}/adataDoubles_sameGroup.h5ad")

HALF_1 = f"K_{GUIDE_GROUP}_1"
HALF_2 = f"K_{GUIDE_GROUP}_2"

moduleGuides = list("GENE_" + guideModulesN.loc[guideModulesN.GuideGroup == GUIDE_GROUP,
                                                "GuideName"] + "_")

adataSinglesTemp = adataSingles[adataSingles.obs.loc[:, moduleGuides + ["GENE_CONTROL_"]].sum(axis=1) == 1]
adataSinglesTemp.obs[HALF_1] = 0
adataSinglesTemp.obs.loc[adataSinglesTemp.obs.loc[:, group1].sum(axis=1) == 1, HALF_1] = 1
adataSinglesTemp.obs[HALF_2] = 0
adataSinglesTemp.obs.loc[adataSinglesTemp.obs.loc[:, group2].sum(axis=1) == 1, HALF_2] = 1

adataDoublesTemp = adataDoubles[adataDoubles.obs[f"K_{GUIDE_GROUP}"] == 1]
adataDoublesTemp.obs[HALF_1] = 1
adataDoublesTemp.obs[HALF_2] = 1

adata = sc.AnnData.concatenate(adataDoublesTemp, adataSinglesTemp)

guideMatrix = adata.obs[[HALF_1, HALF_2]]
expressionMatrix = pd.DataFrame(adata.layers["ClusterResiduals"])
expressionMatrix.columns = adata.var_names
expressionMatrix.index = adata.obs.index

allResp = adata.var_names
my_formula = f"y~{HALF_1}+{HALF_2}+{HALF_1}*{HALF_2}"

print(adata.shape, "|", my_formula)
print(guideMatrix.sum(axis=1).value_counts().to_dict())

## One model per gene

:::{note}
The original looped `seq(1, 1042, 1)` over 1,041 genes, so the last iteration
always failed and was swallowed by `tryCatch`. Looping over the actual column
count drops that silent error and leaves the result unchanged.
:::

In [ ]:
%%R -i guideMatrix,expressionMatrix,my_formula,allResp,OUT_FILE
library(broom)

coefDF <- data.frame()

for (i in seq_len(ncol(expressionMatrix))) {
    tryCatch({
        guideMatrix["y"] <- expressionMatrix[, i]
        myFit <- lm(formula(my_formula), data = guideMatrix)
        myDF  <- data.frame(tidy(myFit))
        myDF$respGene <- allResp[i]
        coefDF <- rbind(coefDF, myDF)
    }, error = function(e) message("gene ", i, " skipped: ", conditionMessage(e)))
}

saveRDS(coefDF, OUT_FILE)
table(coefDF$term)